# 🧍 포즈 랜드마크 (Pose Landmark)

MediaPipe Pose Landmarker를 사용하여人体의 **33개 관절점**을 검출합니다.

```bash
pip install mediapipe opencv-python numpy
```

| 항목 | 내용 |
|------|------|
| **태스크** | `PoseLandmarker` |
| **모델** | `models/pose_landmarker_full.task` |
| **랜드마크 수** | 33개 (코, 어깨, 팔꿈치, 손목, 손가락, 골반, 무릎, 발 등) |
| **좌표** | x, y (정규화 0~1), z (상대 깊이), visibility (가시성) |

---
## 1. 33개 포즈 랜드마크 이해

포즈 랜드마크는 신체의 33개 주요 관절을 검출합니다.

| 인덱스 | 부위 | 인덱스 | 부위 |
|--------|------|--------|------|
| 0 | 코 | 17-22 | 손가락 (좌/우) |
| 1-6 | 눈, 귀 | 23-24 | 골반 (좌/우) |
| 7-10 | 입 | 25-26 | 무릎 (좌/우) |
| 11-12 | 어깨 (좌/우) | 27-28 | 발목 (좌/우) |
| 13-14 | 팔꿈치 (좌/우) | 29-32 | 발뒤꿈치/발끝 (좌/우) |
| 15-16 | 손목 (좌/우) | | |

각 랜드마크는 다음 속성을 가집니다:
- `x`, `y`: 이미지 크기로 정규화된 좌표 (0.0~1.0)
- `z`: 깊이 (코 기준 상대값, 음수 = 카메라에 가까움)
- `visibility`: 해당 관절이 보이는 정도 (0.0~1.0)

In [ ]:
# 포즈 랜드마크 — 정적 이미지 처리
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np
import cv2

# 모델 로드
model_path = 'models/pose_landmarker_full.task'
base_options = mp.tasks.BaseOptions(model_asset_path=model_path)
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5
)

# 테스트 이미지 생성 (실제로는 사람이 있는 이미지 사용)
test_img = np.zeros((480, 640, 3), dtype=np.uint8)
mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=test_img)

# 추론 실행
with vision.PoseLandmarker.create_from_options(options) as landmarker:
    result = landmarker.detect(mp_image)
    
    print(f'감지된 포즈 수: {len(result.pose_landmarks)}')
    print(f'포즈 월드 랜드마크: {len(result.pose_world_landmarks)}')
    print(f'분할 마스크: {result.segmentation_masks if result.segmentation_masks else "없음"}')
    
    if result.pose_landmarks:
        landmarks = result.pose_landmarks[0]
        print(f'\n첫 5개 랜드마크:')
        names = ['코', '왼쪽 눈(안)', '왼쪽 눈(밖)', '오른쪽 눈(안)', '오른쪽 눈(밖)']
        for i, lm in enumerate(landmarks[:5]):
            print(f'  [{i}] {names[i]}: x={lm.x:.3f}, y={lm.y:.3f}, '
                  f'z={lm.z:.3f}, visibility={lm.visibility:.3f}')
    else:
        print('포즈가 감지되지 않았습니다 (테스트 이미지이므로 정상)')

---
## 2. 포즈 랜드마크 시각화

`drawing_utils`와 `PoseLandmarksConnections`를 사용하여
랜드마크와 연결선을 이미지 위에 그립니다.

In [ ]:
# 포즈 랜드마크 시각화 함수
import mediapipe as mp
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import drawing_utils
from mediapipe.tasks.python.vision import drawing_styles
import numpy as np
import cv2

def draw_pose_landmarks(image, pose_landmarks):
    """이미지 위에 포즈 랜드마크를 그립니다."""
    # 랜드마크를 NormalizedLandmark 리스트로 변환
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image.copy())
    
    # 연결선 그리기
    for connection in mp.tasks.vision.PoseLandmarksConnections.POSE_CONNECTIONS:
        start_idx, end_idx = connection
        if start_idx < len(pose_landmarks) and end_idx < len(pose_landmarks):
            start = pose_landmarks[start_idx]
            end = pose_landmarks[end_idx]
            h, w = image.shape[:2]
            pt1 = (int(start.x * w), int(start.y * h))
            pt2 = (int(end.x * w), int(end.y * h))
            cv2.line(image, pt1, pt2, (0, 255, 0), 2)
    
    # 랜드마크 점 그리기
    h, w = image.shape[:2]
    for idx, lm in enumerate(pose_landmarks):
        if lm.visibility > 0.5:  # 가시성이 낮으면 건너뜀
            pt = (int(lm.x * w), int(lm.y * h))
            # 주요 관절은 더 크게
            if idx in [0, 11, 12, 15, 16, 23, 24, 27, 28]:
                cv2.circle(image, pt, 8, (0, 0, 255), -1)
            else:
                cv2.circle(image, pt, 4, (255, 0, 0), -1)
    
    return image

# 랜드마크 이름 매핑 (주요 포인트만)
POSE_NAMES = {
    0: '코', 11: '왼쪽 어깨', 12: '오른쪽 어깨',
    13: '왼쪽 팔꿈치', 14: '오른쪽 팔꿈치',
    15: '왼쪽 손목', 16: '오른쪽 손목',
    23: '왼쪽 골반', 24: '오른쪽 골반',
    25: '왼쪽 무릎', 26: '오른쪽 무릎',
    27: '왼쪽 발목', 28: '오른쪽 발목',
}
print('포즈 시각화 함수 준비 완료')
print(f'주요 랜드마크 {len(POSE_NAMES)}개 이름 매핑됨')

---
## 3. 웹캠 실시간 포즈 검출

웹캠에서 실시간으로 포즈를 검출하고 시각화합니다.
`LIVE_STREAM` 모드와 콜백 함수를 사용합니다.

In [ ]:
# 웹캠 실시간 포즈 검출 (이 셀을 실행하려면 웹캠 필요)
# 이 코드는 참고용 — 웹캠이 있는 환경에서 실행하세요

# import mediapipe as mp
# from mediapipe.tasks import python
# from mediapipe.tasks.python import vision
# import cv2
# import numpy as np
# import time

# model_path = 'models/pose_landmarker_full.task'
# base_options = mp.tasks.BaseOptions(model_asset_path=model_path)
# options = vision.PoseLandmarkerOptions(
#     base_options=base_options,
#     running_mode=vision.RunningMode.LIVE_STREAM,
#     num_poses=1,
#     min_pose_detection_confidence=0.5,
#     result_callback=lambda result, output_image, timestamp_ms: None
# )

# cap = cv2.VideoCapture(0)
# timestamp = 0

# with vision.PoseLandmarker.create_from_options(options) as landmarker:
#     while cap.isOpened():
#         ret, frame = cap.read()
#         if not ret: break
#         
#         mp_image = mp.Image(mp.ImageFormat.SRGB, frame)
#         timestamp = int(time.time() * 1000)
#         landmarker.detect_async(mp_image, timestamp)
#         
#         # 최근 결과 가져오기
#         if landmarker.latest_results and landmarker.latest_results.pose_landmarks:
#             frame = draw_pose_landmarks(frame, landmarker.latest_results.pose_landmarks[0])
#         
#         cv2.imshow('Pose', frame)
#         if cv2.waitKey(1) & 0xFF == ord('q'): break
# cap.release()
# cv2.destroyAllWindows()
print('웹캠 코드는 주석 처리됨 — 웹캠 환경에서 주석 해제 후 실행')

---
## 4. 실전: 자세 분석 — 어깨 수평도 검사

어깨 랜드마크(11, 12)의 y좌표를 비교하여 어깨가 수평인지 확인합니다.
자세 교정, 스쿼트 자세 분석 등에 활용할 수 있습니다.

In [ ]:
# 자세 분석 함수 — 어깨 수평도
import numpy as np

def analyze_posture(pose_landmarks):
    """어깨 수평도와 자세를 분석합니다."""
    if not pose_landmarks or len(pose_landmarks) < 25:
        return None
    
    left_shoulder = pose_landmarks[11]
    right_shoulder = pose_landmarks[12]
    left_hip = pose_landmarks[23]
    right_hip = pose_landmarks[24]
    
    # 어깨 수평도 (y좌표 차이)
    shoulder_diff = abs(left_shoulder.y - right_shoulder.y)
    
    # 골반 수평도
    hip_diff = abs(left_hip.y - right_hip.y)
    
    # 어깨 너비
    shoulder_width = abs(left_shoulder.x - right_shoulder.x)
    
    # 판정
    posture_status = '양호'
    if shoulder_diff > 0.05:
        posture_status = '어깨 틀어짐'
    elif shoulder_diff > 0.03:
        posture_status = '약간 틀어짐'
    
    return {
        'shoulder_diff': shoulder_diff,
        'hip_diff': hip_diff,
        'shoulder_width': shoulder_width,
        'status': posture_status,
        'left_shoulder_vis': left_shoulder.visibility,
        'right_shoulder_vis': right_shoulder.visibility,
    }

# 테스트 (가상 랜드마크)
class FakeLandmark:
    def __init__(self, x, y, z=0, visibility=1.0):
        self.x, self.y, self.z, self.visibility = x, y, z, visibility

# 수평인 어깨
flat_pose = [FakeLandmark(0.5, 0.3)] * 33
flat_pose[11] = FakeLandmark(0.4, 0.3, visibility=0.9)
flat_pose[12] = FakeLandmark(0.6, 0.31, visibility=0.9)
flat_pose[23] = FakeLandmark(0.45, 0.6, visibility=0.8)
flat_pose[24] = FakeLandmark(0.55, 0.61, visibility=0.8)

result = analyze_posture(flat_pose)
print('수평 자세 분석:')
for k, v in result.items():
    print(f'  {k}: {v}')

# 틀어진 어깨
tilted_pose = list(flat_pose)
tilted_pose[11] = FakeLandmark(0.4, 0.25, visibility=0.9)  # 왼쪽 어깨가 높음
tilted_pose[12] = FakeLandmark(0.6, 0.35, visibility=0.9)
result2 = analyze_posture(tilted_pose)
print(f'\n틀어진 자세 분석:')
print(f'  status: {result2["status"]}')
print(f'  shoulder_diff: {result2["shoulder_diff"]:.3f}')

---
## 5. 실전: 팔 각도 측정

어깨-팔꿈치-손목 랜드마크를 사용하여 팔 각도를 계산합니다.
운동 자세 분석, 수어 인식 등에 활용할 수 있습니다.

In [ ]:
# 팔 각도 측정 함수
import math

def calculate_angle(landmarks, idx_a, idx_b, idx_c):
    """세 점 사이의 각도를 계산합니다 (도 단위).
    idx_a - idx_b - idx_c 순서로 각도 측정
    """
    a = np.array([landmarks[idx_a].x, landmarks[idx_a].y])
    b = np.array([landmarks[idx_b].x, landmarks[idx_b].y])
    c = np.array([landmarks[idx_c].x, landmarks[idx_c].y])
    
    # 벡터 계산
    ba = a - b
    bc = c - b
    
    # 코사인 각도
    cosine = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
    cosine = np.clip(cosine, -1.0, 1.0)
    angle = np.degrees(np.arccos(cosine))
    
    return angle

# 테스트: 가상 랜드마크로 팔 각도 계산
# 180도 = 팔을 쭉 편 상태
# 90도 = 팔을 구부린 상태
straight_arm = list(flat_pose)
straight_arm[11] = FakeLandmark(0.3, 0.3)   # 왼쪽 어깨
straight_arm[13] = FakeLandmark(0.4, 0.4)   # 왼쪽 팔꿈치
straight_arm[15] = FakeLandmark(0.5, 0.5)   # 왼쪽 손목

angle = calculate_angle(straight_arm, 11, 13, 15)
print(f'팔을 편 상태의 각도: {angle:.1f}도 (180도에 가까워야 함)')

# 팔을 90도로 구부린 경우
bent_arm = list(flat_pose)
bent_arm[11] = FakeLandmark(0.3, 0.3)       # 어깨
bent_arm[13] = FakeLandmark(0.3, 0.45)      # 팔꿈치 (아래로)
bent_arm[15] = FakeLandmark(0.45, 0.45)     # 손목 (옆으로)

angle2 = calculate_angle(bent_arm, 11, 13, 15)
print(f'팔을 구부린 상태의 각도: {angle2:.1f}도 (90도에 가까워야 함)')

print('\n각도 측정 응용:')
print('  - 팔꿈치 각도: 어깨(11/12) - 팔꿈치(13/14) - 손목(15/16)')
print('  - 무릎 각도: 골반(23/24) - 무릎(25/26) - 발목(27/28)')
print('  - 스쿼트 깊이 측정, 팔 굽혀펴기 자세 분석 등에 활용')

---
## 🎯 정리

| 기능 | 코드 | 설명 |
|------|------|------|
| 포즈 검출 | `PoseLandmarker.detect(image)` | 33개 랜드마크 검출 |
| 랜드마크 | `result.pose_landmarks[0]` | x, y, z, visibility |
| 월드 좌표 | `result.pose_world_landmarks` | 실제 3D 좌표 (미터 단위) |
| 분할 마스크 | `result.segmentation_masks` | 배경/전경 분할 |
| 가시성 | `landmark.visibility` | 0~1, 낮으면 가려진 관절 |
| 각도 계산 | `calculate_angle(lm, a, b, c)` | 세 점 사이 각도 |

**주요 랜드마크 인덱스:**
| 인덱스 | 부위 | 응용 |
|--------|------|------|
| 0 | 코 | 시선 방향 추정 |
| 11, 12 | 어깨 | 자세 수평도, 어깨 너비 |
| 13, 14 | 팔꿈치 | 팔 각도 측정 |
| 15, 16 | 손목 | 손 위치 추적 |
| 23, 24 | 골반 | 몸통 중심, 하체 안정성 |
| 25, 26 | 무릎 | 스쿼트/런지 자세 분석 |
| 27, 28 | 발목 | 균형, 보행 분석 |

---
## 🎯 연습 문제

1. **스쿼트 카운터**: 무릎 각도(골반-무릎-발목)가 90도 이하가 되면 '스쿼트 1회'로 카운트하는 함수를 작성하세요.

2. **얼굴 방향 추정**: 코(0), 왼쪽 귀(7), 오른쪽 귀(8) 랜드마크를 사용하여 얼굴이 어느 방향을 향하는지 판단하는 함수를 작성하세요.

3. **몸통 기울기**: 왼쪽 어깨(11)와 오른쪽 골반(24)을 연결하는 선의 기울기로 몸통이 좌우로 기울어진 정도를 계산하세요.

4. **포즈 분류**: 서기, 앉기, 눕기, 구부리기 등 기본 자세를 랜드마크 좌표로 분류하는 간단한 규칙 기반 함수를 작성하세요. (힌트: 골반-무릎 각도, 머리 높이 등 활용)